# OpenSSL: keys and file formats

Run cells from top to bottom. Generated keys are random, so their values differ between runs.

Written answers (**Your answer** cells) are not checked automatically – after you finish a task, verify them against the notebook in `solutions/`.

In [ ]:
!openssl version

## Generate the working files
Run this cell once. It creates RSA and EC keys in PEM and DER. Encryption, signatures and key exchange follow in `05_rsa_from_scratch.ipynb` and `06_ec_from_scratch.ipynb`.

In [ ]:
!openssl genpkey -algorithm RSA -outform PEM -pkeyopt rsa_keygen_bits:2048 -pkeyopt rsa_keygen_pubexp:3 -out RSA_2048_priv.pem
!openssl pkey -inform PEM -outform DER -in RSA_2048_priv.pem -out RSA_2048_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in RSA_2048_priv.pem -out RSA_2048_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in RSA_2048_pub.pem -out RSA_2048_pub.der

!openssl genpkey -algorithm EC -outform PEM -pkeyopt ec_paramgen_curve:P-256 -pkeyopt ec_param_enc:named_curve -out EC_256_priv.pem
!openssl pkey -inform PEM -outform DER -in EC_256_priv.pem -out EC_256_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in EC_256_priv.pem -out EC_256_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in EC_256_pub.pem -out EC_256_pub.der

## Tasks
Useful commands:
- `openssl asn1parse -i -inform DER -in FILE` prints a DER ASN.1 structure.
- `openssl pkey -text -noout -in FILE` prints key details. Add `-pubin` for a public key.

### Task 1: Convert PEM to DER
Run the cell below. It decodes `RSA_2048_pub.pem` and verifies that the result is exactly `RSA_2048_pub.der`.

In [ ]:
import base64

pem_lines = open("RSA_2048_pub.pem").read().splitlines()
assert pem_lines[0].startswith("-----BEGIN ") and pem_lines[0].endswith("-----")
assert pem_lines[-1].startswith("-----END ") and pem_lines[-1].endswith("-----")
assert pem_lines[0][11:-5] == pem_lines[-1][9:-5]
der_from_pem = base64.b64decode("".join(pem_lines[1:-1]), validate=True)
der_file = open("RSA_2048_pub.der", "rb").read()

print(der_from_pem == der_file)
assert der_from_pem == der_file

**a)** What is the role of Base64 in the PEM representation?

**b)** What should be verified in the PEM header/footer before sharing the key?

**Your answer** (double-click to edit)

**a)** …

**b)** …

### Task 2: Read a DER RSA public key
Run the cells in order. Start with the ASN.1 structure; the raw bytes come last.

In [ ]:
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der

**a)** Find the algorithm OID (`OBJECT`) and the `BIT STRING` in the output above. At which offsets do they start? The offset is the first number on each line.

Write the two offsets into the code cell below (replace `...`) and run it.

In [ ]:
oid_offset        = ...  # TODO
bit_string_offset = ...  # TODO

der = open("RSA_2048_pub.der", "rb").read()
assert der[oid_offset] == 0x06, "no OBJECT IDENTIFIER tag (06) at oid_offset"
assert der[bit_string_offset] == 0x03, "no BIT STRING tag (03) at bit_string_offset"
print("OK")

In [ ]:
# parses the content of the BIT STRING; uses bit_string_offset from the cell above
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der -strparse {bit_string_offset}

**b)** What two values does the nested structure contain, and which ASN.1 type stores them?

**Your answer** (double-click to edit)

**b)** …

In [ ]:
der = open("RSA_2048_pub.der", "rb").read()
for offset in range(0, len(der), 16):
    chunk = der[offset:offset + 16]
    print(f"{offset:03}: {chunk.hex(' ')}")

**c)** Locate the modulus and the public exponent in the hex dump above. Each of them is encoded as tag, length, value.

Write your findings into the code cell below (replace `...`) and run it.

In [ ]:
modulus_offset = ...    # TODO: offset of the modulus tag in the hex dump
modulus_length = ...    # TODO: number of value bytes of the modulus (decimal)
exponent_tlv   = "..."  # TODO: all bytes of the exponent (tag, length, value) as hex, in the form "0a 01 ff"

assert der[modulus_offset] == 0x02, "no INTEGER tag (02) at modulus_offset"
assert modulus_length == int.from_bytes(der[modulus_offset + 2:modulus_offset + 4], "big"), "modulus length is wrong"
assert bytes.fromhex(exponent_tlv) == der[-3:], "exponent bytes are wrong"
print("OK")

### Task 3: Inspect RSA and EC key details
Run each cell and answer the questions below it.

In [ ]:
!openssl pkey -inform PEM -text -noout -in RSA_2048_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in RSA_2048_pub.pem

**a)** Record the first and last 8 hexadecimal digits of RSA `n`, `e`, `d`, `p`, and `q` (notebook 04 decodes the full values automatically). Explain the leading `00:` byte – compare with the INTEGER encoding on the slides (128 → `00 80`).

Write this into the **Your answer** cell, then fill in the two numbers in the code cell below it.

**Your answer** (double-click to edit)

**a)** n = …, e = …, d = …, p = …, q = …

Why the printed `n` starts with `00:`: …

In [ ]:
n_bits          = ...  # TODO: bit length of n
n_printed_bytes = ...  # TODO: number of bytes OpenSSL prints for n, including the leading 00

from cryptography.hazmat.primitives.serialization import load_pem_private_key, Encoding, PublicFormat
rsa = load_pem_private_key(open("RSA_2048_priv.pem", "rb").read(), None)
n = rsa.public_key().public_numbers().n
assert n_bits == n.bit_length(), "bit length of n is wrong"
assert n_printed_bytes == n.bit_length() // 8 + 1, "number of printed bytes is wrong"
print("OK")

In [ ]:
!openssl pkey -inform PEM -text -noout -in EC_256_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in EC_256_pub.pem

**b)** Record the EC private key, public point, and curve. The public point is stored as `04 ‖ x ‖ y` (`04` = uncompressed point). Why do x and y have this size?

Write this into the **Your answer** cell, then fill in the two numbers in the code cell below it.

**Your answer** (double-click to edit)

**b)** priv = …, pub = …, curve = …

Size of x and y: …

In [ ]:
point_bytes      = ...  # TODO: length of the public point `pub` in bytes
coordinate_bytes = ...  # TODO: length of x (and of y) in bytes

ec = load_pem_private_key(open("EC_256_priv.pem", "rb").read(), None)
point = ec.public_key().public_bytes(Encoding.X962, PublicFormat.UncompressedPoint)
assert point_bytes == len(point), "length of the public point is wrong"
assert coordinate_bytes == ec.curve.key_size // 8, "length of x and y is wrong"
print("OK")

In [ ]:
!openssl asn1parse -i -inform DER -in RSA_2048_priv.der

In [ ]:
!openssl asn1parse -i -inform DER -in EC_256_priv.der

**c)** Which values do the `INTEGER`s of `RSA_2048_priv.der` hold? Then find the RSA private exponent `d` (`INTEGER`), the EC private key (`OCTET STRING`), the curve identifier (`OBJECT`) and the public point (`BIT STRING`) in the two outputs above.

List the `INTEGER`s in the **Your answer** cell, then write the four offsets into the code cell below it.

**Your answer** (double-click to edit)

**c)** RSA `INTEGER`s in order: …

In [ ]:
rsa_d_offset    = ...  # TODO: RSA private exponent d in RSA_2048_priv.der
ec_priv_offset  = ...  # TODO: EC private key (OCTET STRING) in EC_256_priv.der
ec_curve_offset = ...  # TODO: curve identifier (OBJECT)
ec_point_offset = ...  # TODO: public point (BIT STRING)

rsa_der = open("RSA_2048_priv.der", "rb").read()
ec_der = open("EC_256_priv.der", "rb").read()
d = rsa.private_numbers().d
d = d.to_bytes((d.bit_length() + 7) // 8, "big")
priv = ec.private_numbers().private_value.to_bytes(32, "big")
assert rsa_der[rsa_d_offset] == 0x02 and d in rsa_der[rsa_d_offset:rsa_d_offset + 5 + len(d)], "d does not start at rsa_d_offset"
assert ec_der[ec_priv_offset] == 0x04 and priv in ec_der[ec_priv_offset:ec_priv_offset + 2 + len(priv)], "the EC private key does not start at ec_priv_offset"
assert ec_der[ec_curve_offset] == 0x06, "no OBJECT IDENTIFIER tag (06) at ec_curve_offset"
assert ec_der[ec_point_offset] == 0x03, "no BIT STRING tag (03) at ec_point_offset"
print("OK")